# Hyperparameter Tuning with Optuna

In this notebook, we focus on optimizing the hyperparameters of the selected models (Gradient Boosting and CatBoost) to maximize their performance. We utilize **Optuna**, an automatic hyperparameter optimization framework, to search for the best set of parameters efficiently.

**Objectives:**
1.  **Hyperparameter Search:** Define search spaces and objectives for Gradient Boosting and CatBoost models.
2.  **Optimization:** Run Optuna studies to maximize the **ROC-AUC** score.
3.  **Best Parameters:** Identify and record the optimal hyperparameters for the final model training.

In [ ]:
# Import necessary libraries
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
import optuna

# Add the project root directory to sys.path to allow importing from 'src'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import the custom preprocessor after adding path
from src.ml.preprocessor import CustomerChurnPreprocessor, build_encoder

In [ ]:
# Load and preprocess the training data
train_data = pd.read_csv('../data/processed/train.csv')

In [ ]:
# Preprocess the data using the custom preprocessor
preprocessor = CustomerChurnPreprocessor()
train_data_processed = preprocessor.preprocess(train_data)

In [ ]:
# Separate features and target variable
X_train = train_data_processed.drop('Exited', axis=1)
y_train = train_data_processed['Exited']

print(f"Training Set Size: {X_train.shape}")

In [ ]:
def objective_gb(trial):
    """
    Optuna objective function for Gradient Boosting Classifier.
    Arg:
        trial: A trial object of Optuna.
    Returns:
        float: Mean ROC-AUC score from cross-validation.
    """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'random_state': 42
    }
    
    pipeline = make_pipeline(
        build_encoder(),
        VarianceThreshold(threshold=0),
        MinMaxScaler(),
        GradientBoostingClassifier(**params)
    )
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    return scores.mean()


In [ ]:
def objective_cat(trial):
    """
    Optuna objective function for CatBoost Classifier.
    Arg:
        trial: A trial object of Optuna.
    Returns:
        float: Mean ROC-AUC score from cross-validation.
    """
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'random_state': 42,
        'verbose': 0,  
        'allow_writing_files': False
    }
    
    pipeline = make_pipeline(
        build_encoder(),
        VarianceThreshold(threshold=0),
        MinMaxScaler(),
        CatBoostClassifier(**params)
    )
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    return scores.mean()

In [ ]:
print("1. Starting Gradient Boosting Optimization...")
study_gb = optuna.create_study(direction='maximize')
study_gb.optimize(objective_gb, n_trials=50)

In [ ]:
print("\n2. Starting CatBoost Optimization...")
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=50)

In [ ]:
print("-" * 50)
print("RESULTS")
print("-" * 50)
print(f"Gradient Boosting Best Score (ROC-AUC): {study_gb.best_value:.4f}")
print("Best Parameters:", study_gb.best_params)
print("-" * 50)
print(f"CatBoost Best Score (ROC-AUC): {study_cat.best_value:.4f}")
print("Best Parameters:", study_cat.best_params)
print("-" * 50)